In [2]:
 %pip install nba_api
 %pip install pandas
 %pip install scikit-learn
 %pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.0/319.0 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 764.2/764.2 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 18.9 MB/s eta 0:00:00


*Importación de Librerias*

In [8]:
import pandas as pd
import time
import joblib
import mlflow
import mlflow.sklearn
from nba_api.stats.endpoints import leaguestandings
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

Recolección de datos

In [9]:
def obtener_datos_entrenamiento(anios):
    data_frames = []
    for anio in anios:
        season_str = f"{anio}-{str(anio+1)[-2:]}"
        try:
            standings = leaguestandings.LeagueStandings(season=season_str)
            df = standings.get_data_frames()[0]
            # Target: Rank <= 8 es Playoffs
            df['Playoffs'] = df['PlayoffRank'].apply(lambda x: 1 if x <= 8 else 0)
            data_frames.append(df)
            time.sleep(0.6)
        except Exception as e:
            print(f"Error en {season_str}: {e}")
    return pd.concat(data_frames, ignore_index=True)

# Cargar datos
anios = range(2015, 2023)
df_full = obtener_datos_entrenamiento(anios)

# Variables
features = ['WinPCT', 'PointsPG', 'OppPointsPG', 'PlayoffRank']
X = df_full[features]
y = df_full['Playoffs']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
results = {}

Mlflow Setup

In [12]:
mlflow.set_experiment("NBA_Playoff_Predicciones_LogReg")

<Experiment: artifact_location='/content/mlruns/1', creation_time=1765605884177, experiment_id='1', last_update_time=1765605884177, lifecycle_stage='active', name='NBA_Playoff_Predicciones_LogReg', tags={}>

Modelo 1: Regresion Logistica

In [17]:
with mlflow.start_run(run_name="Regresion_Logistica"):
    print("Entrenando Regresión Logística...")

    pipeline_lr = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=42))
    ])

    # GridSearch para optimizar hiperparámetros
    param_grid = {'clf__C': [0.1, 1, 10], 'clf__solver': ['liblinear', 'lbfgs']}
    grid_lr = GridSearchCV(pipeline_lr, param_grid, cv=5, scoring='accuracy')
    grid_lr.fit(X_train, y_train)

    best_lr = grid_lr.best_estimator_
    acc_lr = accuracy_score(y_test, best_lr.predict(X_test))

    # Loguear en MLflow
    mlflow.log_params(grid_lr.best_params_)
    mlflow.log_metric("accuracy", acc_lr)
    mlflow.sklearn.log_model(best_lr, "model_logreg")

    results['LogReg'] = {'acc': acc_lr, 'model': best_lr}
    print(f"   -> Accuracy LogReg: {acc_lr:.4f}")

Entrenando Regresión Logística...


2025/12/13 06:17:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


   -> Accuracy LogReg: 1.0000


Modelo 2: Naive Bayes

In [18]:
with mlflow.start_run(run_name="Naive_Bayes"):
    print("Entrenando Naive Bayes...")

    pipeline_nb = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GaussianNB()) # Naive Bayes no suele requerir mucho tuning
    ])

    pipeline_nb.fit(X_train, y_train)
    acc_nb = accuracy_score(y_test, pipeline_nb.predict(X_test))

    # Loguear en MLflow
    mlflow.log_metric("accuracy", acc_nb)
    mlflow.sklearn.log_model(pipeline_nb, "model_nb")

    results['NaiveBayes'] = {'acc': acc_nb, 'model': pipeline_nb}
    print(f"   -> Accuracy Naive Bayes: {acc_nb:.4f}")

2025/12/13 06:18:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Entrenando Naive Bayes...
   -> Accuracy Naive Bayes: 0.9792


In [19]:
if results['Regresion Logistica']['accuracy'] > results['Naive Bayes']['accuracy']:
    campeon = results['Regresion Logistica']['model']
    nombre_campeon = "Regresión Logística"
else:
    campeon = results['Naive Bayes']['model']
    nombre_campeon = "Naive Bayes"

print(f"\n--- ¡MODELO CAMPEÓN SELECCIONADO: {nombre_campeon} ---")
print(f"Accuracy de Regresión Logística: {results['Regresion Logistica']['accuracy']:.4f}")
print(f"Accuracy de Naive Bayes: {results['Naive Bayes']['accuracy']:.4f}")

# Guardar el modelo campeón en joblib
nombre_archivo = 'modelo_nba.joblib'
joblib.dump(campeon, nombre_archivo)
print(f"\nEl modelo '{nombre_campeon}' fue guardado como '{nombre_archivo}' para tu API.")


--- ¡MODELO CAMPEÓN SELECCIONADO: Regresión Logística ---
Accuracy de Regresión Logística: 1.0000
Accuracy de Naive Bayes: 0.9792

El modelo 'Regresión Logística' fue guardado como 'modelo_nba.joblib' para tu API.
